# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MahboobAli1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# ## Unit of Analysis + Time Window

# - One row represents the daily search performance of one content item for one client.
# - The time window used in this notebook is March 2026 (month = '2026-03').

In [1]:
!pip -q install duckdb datasets huggingface_hub pandas pyarrow fsspec

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [3]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print(f"Total files: {len(files)}")
for f in files[:30]:
    print(f)

Total files: 24
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_

In [4]:
import duckdb

con = duckdb.connect()

In [7]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

print(march_file)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [12]:
df = con.sql(f"""
SELECT *
FROM read_parquet('{march_file}')
""").df()

con.register("performance", df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS records
FROM performance
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,records


In [14]:
con.sql("""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM performance
""").df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [15]:
con.sql("""
SELECT
COUNT(*) AS available_rows
FROM performance
WHERE gsc_data_available IS TRUE
""").df()

,available_rows
0,3611061


In [9]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   report_date               5 non-null      datetime64[us]
 1   client_hash_id            5 non-null      object        
 2   content_hash_id           5 non-null      object        
 3   client_has_gsc            5 non-null      bool          
 4   client_has_ga4            5 non-null      bool          
 5   gsc_data_available        5 non-null      bool          
 6   ga4_data_available        0 non-null      boolean       
 7   gsc_impressions           5 non-null      int64         
 8   gsc_clicks                5 non-null      int64         
 9   gsc_sum_position          5 non-null      int64         
 10  gsc_avg_position          5 non-null      float64       
 11  ga4_pageviews             0 non-null      Int64         
 12  ga4_sessions              

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# ## Fields

# ### Features
# - gsc_impressions
# - gsc_avg_position
# - gsc_clicks
# - client_has_gsc
# - gsc_data_available

# ### Label
# - gsc_clicks (used as a proxy target for this exercise)

# ### Context
# - report_date
# - client_hash_id
# - content_hash_id
# - month

# ### Excluded
# - All GA4 columns because this client has no GA4 data available.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
con.register("performance", df)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# | Feature | Available when? |
# |----------|-----------------|
# | gsc_impressions | Known before prediction because it is historical search data. |
# | gsc_avg_position | Known before prediction because ranking is already observed. |
# | client_has_gsc | Known before prediction because it is client metadata. |
# | gsc_data_available | Known before prediction because it indicates data availability. |
# # | report_date | Known before prediction because the observation date already exists. |

In [ ]:
# ## Data Limits

# - This notebook uses only one month (March 2026).
# - GA4 data is unavailable for this client, so user engagement cannot be analyzed.
# - The dataset contains anonymized IDs, making it impossible to identify real clients or pages.

In [ ]:
# ==========================================================
# ML-04 — Search Intelligence Data Contract
# Section 1–6 Complete Code
# ==========================================================

# ==========================================================
# 1. Install Required Libraries
# ==========================================================

!pip -q install duckdb datasets huggingface_hub pandas pyarrow fsspec scikit-learn

# ==========================================================
# 2. Load Hugging Face Token
# ==========================================================

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

# ==========================================================
# 3. Download March 2026 Dataset
# ==========================================================

from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

print(march_file)

# ==========================================================
# 4. Load Dataset into DuckDB
# ==========================================================

import duckdb
import pandas as pd

con = duckdb.connect()

df = con.sql(f"""
SELECT *
FROM read_parquet('{march_file}')
""").df()

print(df.head())

print("\nColumns:")
print(df.columns.tolist())

print("\nDataset Info:")
print(df.info())

con.register("performance", df)

# ==========================================================
# 5. Verification Queries
# ==========================================================

print("========================================")
print("Query 1: Verify Grain")
print("========================================")

grain = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS records
FROM performance
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()

display(grain)


print("========================================")
print("Query 2: Row Count & Date Span")
print("========================================")

summary = con.sql("""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM performance
""").df()

display(summary)


print("========================================")
print("Query 3: Availability Check")
print("========================================")

availability = con.sql("""
SELECT
COUNT(*) AS available_rows
FROM performance
WHERE gsc_data_available IS TRUE
""").df()

display(availability)

# ==========================================================
# 6. Five Features + Leakage Experiment
# ==========================================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# -----------------------------
# Feature Frame
# -----------------------------

feature_df = df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "client_has_gsc",
        "gsc_data_available",
        "report_date",
        "gsc_clicks"
    ]
].copy()

feature_df = feature_df.dropna()

print("Feature Frame")
display(feature_df.head())

# -----------------------------
# Honest Model
# -----------------------------

print("\n==============================")
print("Honest Model")
print("==============================")

X = feature_df[
    [
        "gsc_impressions",
        "gsc_avg_position"
    ]
]

y = feature_df["gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

honest_score = r2_score(y_test, pred)

print("Honest R² Score:", honest_score)

# -----------------------------
# Leakage
# -----------------------------

print("\n==============================")
print("Leakage Model")
print("==============================")

feature_df["leaked_clicks"] = feature_df["gsc_clicks"]

X = feature_df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "leaked_clicks"
    ]
]

y = feature_df["gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

leaked_score = r2_score(y_test, pred)

print("Leaked R² Score:", leaked_score)

# -----------------------------
# Remove Leakage
# -----------------------------

feature_df.drop(columns=["leaked_clicks"], inplace=True)

print("\nLeakage feature removed successfully.")

print("\n===================================")
print("Comparison")
print("===================================")

print("Honest Score :", honest_score)
print("Leaked Score :", leaked_score)

Token loaded: True
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True           False                True                <NA>   
1            True           False                True                <NA>   
2            True           False                True                <NA>   
3            True           False                True                <NA>   
4            True           False                True                <NA>   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               20           0                67  ...     

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,records


...

Query 3: Availability Check


,available_rows
0,3611061


Feature Frame


,gsc_impressions,gsc_avg_position,client_has_gsc,gsc_data_available,report_date,gsc_clicks
0,20,3.350000,True,True,2026-03-01,0
1,1,0.000000,True,True,2026-03-01,0
2,125,4.928000,True,True,2026-03-01,1
3,7,4.000000,True,True,2026-03-01,0
4,11,2.272727,True,True,2026-03-01,0



Honest Model


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.